# Zerobus producer
Pushes order events straight into the Unity Catalog table `events_bronze` over gRPC. There is no broker, topic or consumer between the producer and the table.

`event_id` and the payload are derived from `producer_id` and `seq`, so a replayed range produces identical records. That models a producer that restarts from its last checkpoint: Zerobus is at-least-once and the replay lands in bronze as duplicates.

A flush means the records are durable, not that they are queryable yet, so both times are measured.

In [ ]:
%pip install -q databricks-zerobus-ingest-sdk==1.10.0
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("zerobus_client_id", "")
dbutils.widgets.text("secret_scope", "")
dbutils.widgets.text("secret_key", "")
dbutils.widgets.text("zerobus_endpoint", "")
dbutils.widgets.text("workspace_url", "")
dbutils.widgets.text("producer_id", "orders-api")
dbutils.widgets.text("step", "initial")
dbutils.widgets.text("from_seq", "0")
dbutils.widgets.text("to_seq", "0")

target = f"{dbutils.widgets.get('catalog')}.{dbutils.widgets.get('schema')}"
bronze = f"{target}.events_bronze"
producer_id = dbutils.widgets.get("producer_id")
step = dbutils.widgets.get("step")
from_seq = int(dbutils.widgets.get("from_seq"))
to_seq = int(dbutils.widgets.get("to_seq"))

In [ ]:
import datetime as dt
import random
import time

from zerobus.sdk.shared import TableProperties
from zerobus.sdk.sync import ZerobusSdk

START = dt.datetime(2026, 1, 1, tzinfo=dt.timezone.utc)
EVENT_TYPES = ("created", "paid", "shipped")


def build_event(seq):
    rng = random.Random(f"{producer_id}:{seq}")
    moment = START + dt.timedelta(seconds=seq)
    return {
        "event_id": f"{producer_id}-{seq:010d}",
        "producer_id": producer_id,
        "seq": seq,
        "order_id": seq // len(EVENT_TYPES),
        "event_type": EVENT_TYPES[seq % len(EVENT_TYPES)],
        "amount": round(rng.uniform(5, 500), 2),
        "event_time": int(moment.timestamp() * 1_000_000),
    }

In [ ]:
records_before = spark.table(bronze).count()
records = to_seq - from_seq

sdk = ZerobusSdk(dbutils.widgets.get("zerobus_endpoint"), dbutils.widgets.get("workspace_url"))
stream = sdk.create_stream(
    dbutils.widgets.get("zerobus_client_id"),
    dbutils.secrets.get(dbutils.widgets.get("secret_scope"), dbutils.widgets.get("secret_key")),
    TableProperties(bronze),
)

started = time.perf_counter()
try:
    for seq in range(from_seq, to_seq):
        stream.ingest_record_offset(build_event(seq))
    stream.flush()
    durable_seconds = time.perf_counter() - started
finally:
    stream.close()

In [ ]:
deadline = time.perf_counter() + 300
while spark.table(bronze).count() < records_before + records:
    if time.perf_counter() > deadline:
        raise TimeoutError(f"{records} records are durable but not visible in {bronze} after 300 s")
    time.sleep(1)
queryable_seconds = time.perf_counter() - started

metrics = [(producer_id, step, from_seq, to_seq, records, durable_seconds, records / durable_seconds, queryable_seconds)]
(
    spark.createDataFrame(metrics, "producer_id string, step string, from_seq bigint, to_seq bigint, records bigint, durable_seconds double, records_per_second double, queryable_seconds double")
    .selectExpr("*", "current_timestamp() AS finished_at")
    .write.mode("append").saveAsTable(f"{target}.producer_metrics")
)
print(f"{step}: {records} records, durable {durable_seconds:.1f} s, queryable {queryable_seconds:.1f} s, {records / durable_seconds:.0f} records/s")